# NBA Playoff Simulator


## Step 1: Install NBA API, Scikit-Learn, Pandas, Numpy, Rich, and Torch

In [ ]:
pip install nba_api scikit-learn pandas numpy rich torch

**Workflow — run cells top to bottom once, then re-run only what you need:**

| Cell | What it does | Re-run when… |
|------|-------------|--------------|
| 1 | Imports & helpers | Never (unless env changes) |
| 2 | Fetch historical playoff data for ML training | Never (slow, cached to disk) |
| 3 | Train ML model on historical data | Never (saved to disk) |
| 4 | Enter your 16 teams + fetch their season stats | Changing teams or season |
| 5 | Simulate playoffs | Any time — fast, no API calls |

In [1]:
import warnings, time, random, os, pickle
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from itertools import combinations

from nba_api.stats.endpoints import teamgamelog, leaguegamefinder
from nba_api.stats.static import teams as nba_teams_static

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

ALL_TEAMS   = {t["abbreviation"]: t for t in nba_teams_static.get_teams()}
ID_TO_ABBR  = {t["id"]: t["abbreviation"] for t in nba_teams_static.get_teams()}

SEASON_MAP  = {
    2016:"2015-16", 2017:"2016-17", 2018:"2017-18", 2019:"2018-19",
    2020:"2019-20", 2021:"2020-21", 2022:"2021-22", 2023:"2022-23",
    2024:"2023-24", 2025:"2024-25",
}
def season_str(y): return SEASON_MAP.get(y, f"{y-1}-{str(y)[2:]}")

# Stat columns we pull from every game log
STAT_COLS = ["PTS","FG_PCT","FG3_PCT","FT_PCT","REB","OREB","DREB",
             "AST","STL","BLK","TOV","PLUS_MINUS"]

# Feature keys used in ML (differences A-B)
FEAT_KEYS = ["WIN_PCT","PTS","FG_PCT","FG3_PCT","FT_PCT",
             "REB","OREB","AST","STL","BLK","TOV","PLUS_MINUS"]

print("✅ Imports done.")

✅ Imports done.


# Cell 2 - Explanation

Pulls regular-season averages + playoff series outcomes for 2016–2024. Takes ~5 min first run, then loads from disk.
    
## Config -

Just setting up the filename where data gets saved, and the list of seasons to pull (2015-16 through 2023-24). This is your historical training data source.

## fetch_season_avgs & fetch_playoff_series

gets averages for the season and playoffs

In [2]:
CACHE_FILE    = "nba_training_data.pkl"
TRAIN_SEASONS = list(range(2016, 2025))   # 2015-16 through 2023-24

def fetch_season_avgs(abbrev, season):
    """Regular-season per-game averages for a team in a given season."""
    team_id = ALL_TEAMS[abbrev]["id"]
    time.sleep(0.65)
    try:
        gl  = teamgamelog.TeamGameLog(team_id=team_id, season=season,
                                      season_type_all_star="Regular Season")
        df  = gl.get_data_frames()[0]
        df["WIN"] = df["WL"].apply(lambda x: 1 if x=="W" else 0)
        avgs = {c: df[c].mean() for c in STAT_COLS if c in df.columns}
        avgs["W"]       = int(df["WIN"].sum())
        avgs["L"]       = int(len(df) - avgs["W"])
        avgs["WIN_PCT"] = avgs["W"] / max(len(df), 1)
        return avgs
    except Exception as e:
        print(f"  ⚠ {abbrev} {season}: {e}")
        return {}

def fetch_playoff_series(season):
    """
    Returns list of dicts: {winner_abbr, loser_abbr, winner_avgs, loser_avgs}
    by scraping LeagueGameFinder for playoff games and inferring series winners.
    """
    time.sleep(0.65)
    try:
        finder = leaguegamefinder.LeagueGameFinder(
            season_nullable=season,
            season_type_nullable="Playoffs",
            league_id_nullable="00"
        )
        df = finder.get_data_frames()[0]
    except Exception as e:
        print(f"  ⚠ playoff fetch {season}: {e}")
        return []

    if df.empty:
        return []

    df["WIN"] = df["WL"].apply(lambda x: 1 if x=="W" else 0)

    # Identify series by unique (TEAM_ID, opponent) pairs
    # MATCHUP looks like "BOS vs. MIA" or "BOS @ MIA"
    def opp_from_matchup(matchup, team_abbr):
        parts = matchup.replace("@","vs.").split("vs.")
        for p in parts:
            p = p.strip()
            if p != team_abbr:
                return p
        return None

    df["ABBREV"] = df["TEAM_ID"].map(ID_TO_ABBR)
    df["OPP"]    = df.apply(lambda r: opp_from_matchup(r["MATCHUP"], r["ABBREV"]), axis=1)

    # Group by team pair, count wins
    series_records = {}
    for _, row in df.iterrows():
        a, b = row["ABBREV"], row["OPP"]
        if not a or not b:
            continue
        key = tuple(sorted([a, b]))
        series_records.setdefault(key, {a: 0, b: 0})
        if row["WIN"] == 1:
            series_records[key][a] = series_records[key].get(a, 0) + 1

    series_list = []
    for (a, b), wins in series_records.items():
        wa, wb = wins.get(a, 0), wins.get(b, 0)
        if wa + wb < 4:          # not enough games — skip
            continue
        winner = a if wa > wb else b
        loser  = b if wa > wb else a
        series_list.append({"winner": winner, "loser": loser,
                             "winner_wins": max(wa,wb), "loser_wins": min(wa,wb)})
    return series_list


if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "rb") as f:
        training_rows = pickle.load(f)
    print(f"✅ Loaded {len(training_rows)} historical series from cache.")
else:
    training_rows = []
    season_avgs_cache = {}   # (abbrev, season) -> avgs dict

    for year in TRAIN_SEASONS:
        season = season_str(year)
        print(f"\n📅 Season {season}")
        series = fetch_playoff_series(season)
        print(f"   Found {len(series)} playoff series")

        for s in series:
            for role in ["winner", "loser"]:
                key = (s[role], season)
                if key not in season_avgs_cache:
                    print(f"   Fetching reg-season avgs: {s[role]}")
                    season_avgs_cache[key] = fetch_season_avgs(s[role], season)

            row = {
                "season":       season,
                "winner":       s["winner"],
                "loser":        s["loser"],
                "winner_avgs":  season_avgs_cache.get((s["winner"], season), {}),
                "loser_avgs":   season_avgs_cache.get((s["loser"],  season), {}),
            }
            if row["winner_avgs"] and row["loser_avgs"]:
                training_rows.append(row)

    with open(CACHE_FILE, "wb") as f:
        pickle.dump(training_rows, f)
    print(f"\n✅ Fetched & cached {len(training_rows)} historical series.")


📅 Season 2015-16
   Found 15 playoff series
   Fetching reg-season avgs: CLE
   Fetching reg-season avgs: GSW
   Fetching reg-season avgs: OKC
   Fetching reg-season avgs: TOR
   Fetching reg-season avgs: MIA
   Fetching reg-season avgs: SAS
   Fetching reg-season avgs: POR
   Fetching reg-season avgs: ATL
   Fetching reg-season avgs: IND
   Fetching reg-season avgs: CHA
   Fetching reg-season avgs: LAC
   Fetching reg-season avgs: BOS
   Fetching reg-season avgs: HOU
   Fetching reg-season avgs: DAL
   Fetching reg-season avgs: MEM
   Fetching reg-season avgs: DET

📅 Season 2016-17
   Found 15 playoff series
   Fetching reg-season avgs: GSW
   Fetching reg-season avgs: CLE
   Fetching reg-season avgs: BOS
   Fetching reg-season avgs: SAS
   Fetching reg-season avgs: WAS
   Fetching reg-season avgs: HOU
   Fetching reg-season avgs: UTA
   Fetching reg-season avgs: TOR
   Fetching reg-season avgs: LAC
   Fetching reg-season avgs: CHI
   Fetching reg-season avgs: ATL
   Fetching reg-sea

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
import pickle

MODEL_FILE = "nba_model.pkl"

def build_feature(avgs_a, avgs_b):
    """
    Feature vector = (A - B) for every stat key.
    Positive  → A is better in that dimension.
    TOV: lower is better, so negative diff means A has fewer turnovers (good for A).
    """
    return np.array([avgs_a.get(k, 0) - avgs_b.get(k, 0) for k in FEAT_KEYS],
                    dtype=float)

# Build balanced dataset: both orientations of every series
X_train, y_train = [], []

for row in training_rows:
    wa = row["winner_avgs"]
    la = row["loser_avgs"]
    if not wa or not la:
        continue
    f_wl = build_feature(wa, la)   # winner - loser  → label 1
    f_lw = build_feature(la, wa)   # loser  - winner → label 0
    if np.isnan(f_wl).any() or np.isnan(f_lw).any():
        continue
    X_train.extend([f_wl, f_lw])
    y_train.extend([1, 0])

X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"Training samples : {len(X_train)}")
print(f"Features per row : {X_train.shape[1]}")
print(f"Features used    : {FEAT_KEYS}")
print(f"Label balance    : {y_train.mean()*100:.0f}% class-1 (expected 50%)")

# Scale
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

# Logistic Regression — C controls regularization strength.
# Lower C = more regularization = less overconfident predictions.
# C=0.1 is appropriate for ~120 real samples and 12 features.
base_model = LogisticRegression(
    C=0.1,             # regularization — key to keeping probs calibrated
    max_iter=1000,
    random_state=42,
    solver="lbfgs"
)

# Wrap with Platt scaling for well-calibrated probabilities
model = CalibratedClassifierCV(base_model, cv=5, method="sigmoid")

# Cross-validate
cv_scores = cross_val_score(base_model, X_scaled, y_train, cv=5, scoring="accuracy")
print(f"\n5-fold CV accuracy: {cv_scores.mean()*100:.1f}% ± {cv_scores.std()*100:.1f}%")
print("(58-65% is realistic — NBA playoffs have genuine upsets)")

model.fit(X_scaled, y_train)

# Sanity check: what probabilities does it actually output?
sample_probs = model.predict_proba(X_scaled[:20])[:, 1]
print(f"\nSample output probs (first 20 rows): {np.round(sample_probs, 2)}")
print(f"Mean prob: {sample_probs.mean():.2f}  Min: {sample_probs.min():.2f}  Max: {sample_probs.max():.2f}")
print("(Should see spread from ~0.40 to ~0.65, not all 0.90+)")

# Feature importance via LR coefficients
coefs = pd.Series(base_model.fit(X_scaled, y_train).coef_[0], index=FEAT_KEYS)
coefs_sorted = coefs.abs().sort_values(ascending=False)
print("\nFeature importance (by absolute coefficient):")
for feat, val in coefs_sorted.items():
    direction = "↑ favors A" if coefs[feat] > 0 else "↓ favors B"
    print(f"  {feat:<15} {val:.3f}  {direction}")

with open(MODEL_FILE, "wb") as f:
    pickle.dump({"model": model, "scaler": scaler}, f)
print(f"\n✅ Model saved to {MODEL_FILE}")


Training samples : 268
Features per row : 12
Features used    : ['WIN_PCT', 'PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'REB', 'OREB', 'AST', 'STL', 'BLK', 'TOV', 'PLUS_MINUS']
Label balance    : 50% class-1 (expected 50%)

5-fold CV accuracy: 72.0% ± 8.5%
(58-65% is realistic — NBA playoffs have genuine upsets)

Sample output probs (first 20 rows): [0.12 0.88 0.85 0.15 0.71 0.29 0.6  0.4  0.37 0.63 0.95 0.05 0.73 0.26
 0.67 0.33 0.53 0.47 0.47 0.53]
Mean prob: 0.50  Min: 0.05  Max: 0.95
(Should see spread from ~0.40 to ~0.65, not all 0.90+)

Feature importance (by absolute coefficient):
  WIN_PCT         0.980  ↑ favors A
  FG_PCT          0.270  ↑ favors A
  BLK             0.226  ↓ favors B
  AST             0.211  ↑ favors A
  STL             0.207  ↓ favors B
  FT_PCT          0.189  ↑ favors A
  PTS             0.182  ↑ favors A
  TOV             0.179  ↑ favors A
  REB             0.065  ↓ favors B
  OREB            0.056  ↑ favors A
  FG3_PCT         0.013  ↑ favors A
  PLUS_MINUS   

In [18]:
SEASON_YEAR = 2026
EAST_SEEDS  = ["DET","BOS","NYK","CLE","TOR","ATL","PHI","ORL"]  # seed 1 → 8
WEST_SEEDS  = ["OKC","SAS","DEN","LAL","HOU","MIN","POR","PHX"]  # seed 1 → 8

# ─────────────────────────────────────────────────────────────────────────────
season   = season_str(SEASON_YEAR)
all_teams = EAST_SEEDS + WEST_SEEDS
seed_map  = {t: i+1 for i, t in enumerate(EAST_SEEDS)}
seed_map.update({t: i+1 for i, t in enumerate(WEST_SEEDS)})

def fetch_team_games_full(abbrev, season):
    """Full game log including H2H filtering later."""
    team_id = ALL_TEAMS[abbrev]["id"]
    time.sleep(0.65)
    try:
        gl  = teamgamelog.TeamGameLog(team_id=team_id, season=season,
                                      season_type_all_star="Regular Season")
        df  = gl.get_data_frames()[0]
        df["ABBREV"] = abbrev
        df["WIN"]    = df["WL"].apply(lambda x: 1 if x=="W" else 0)
        return df
    except Exception as e:
        print(f"  ⚠ {abbrev}: {e}")
        return pd.DataFrame()

def game_log_to_avgs(df):
    if df.empty: return {}
    avgs = {c: df[c].mean() for c in STAT_COLS if c in df.columns}
    avgs["W"]       = int(df["WIN"].sum())
    avgs["L"]       = int(len(df) - avgs["W"])
    avgs["WIN_PCT"] = avgs["W"] / max(len(df), 1)
    return avgs

print(f"Fetching {len(all_teams)} teams for {season} ...")
team_data = {}
for abbrev in all_teams:
    print(f"  {abbrev} ...", end=" ")
    df   = fetch_team_games_full(abbrev, season)
    avg  = game_log_to_avgs(df)
    team_data[abbrev] = {"df": df, "avg": avg, "h2h": {}}
    print(f"W:{avg.get('W',0)} L:{avg.get('L',0)} PTS:{avg.get('PTS',0):.1f} +/-:{avg.get('PLUS_MINUS',0):+.1f}")

print("\nComputing H2H stats ...")
for a in all_teams:
    for b in all_teams:
        if a == b: continue
        sub = team_data[a]["df"]
        sub = sub[sub["MATCHUP"].str.contains(b, na=False)]
        team_data[a]["h2h"][b] = game_log_to_avgs(sub)

# Pretty table
print(f"\n{'Team':<6} {'Seed':>4} {'Conf':<5} {'W':>3} {'L':>3} {'PTS':>6} {'FG%':>6} {'3P%':>6} {'REB':>5} {'AST':>5} {'STL':>5} {'BLK':>5} {'TOV':>5} {'+/-':>6}")
print("-"*80)
for conf_label, seeds in [("East", EAST_SEEDS), ("West", WEST_SEEDS)]:
    for i, t in enumerate(seeds):
        a = team_data[t]["avg"]
        print(f"{t:<6} {i+1:>4} {conf_label:<5} "
              f"{int(a.get('W',0)):>3} {int(a.get('L',0)):>3} "
              f"{a.get('PTS',0):>6.1f} {a.get('FG_PCT',0):>6.3f} "
              f"{a.get('FG3_PCT',0):>6.3f} {a.get('REB',0):>5.1f} "
              f"{a.get('AST',0):>5.1f} {a.get('STL',0):>5.1f} "
              f"{a.get('BLK',0):>5.1f} {a.get('TOV',0):>5.1f} "
              f"{a.get('PLUS_MINUS',0):>+6.1f}")
    print()

print("✅ Team data ready.")


Fetching 16 teams for 2025-26 ...
  DET ... W:60 L:22 PTS:117.8 +/-:+0.0
  BOS ... W:56 L:26 PTS:114.9 +/-:+0.0
  NYK ... W:53 L:29 PTS:116.5 +/-:+0.0
  CLE ... W:52 L:30 PTS:119.5 +/-:+0.0
  TOR ... W:46 L:36 PTS:114.6 +/-:+0.0
  ATL ... W:46 L:36 PTS:118.5 +/-:+0.0
  PHI ... W:45 L:37 PTS:115.9 +/-:+0.0
  ORL ... W:45 L:37 PTS:115.7 +/-:+0.0
  OKC ... W:64 L:18 PTS:119.0 +/-:+0.0
  SAS ... W:62 L:20 PTS:119.8 +/-:+0.0
  DEN ... W:54 L:28 PTS:122.1 +/-:+0.0
  LAL ... W:53 L:29 PTS:116.3 +/-:+0.0
  HOU ... W:52 L:30 PTS:115.2 +/-:+0.0
  MIN ... W:49 L:33 PTS:118.0 +/-:+0.0
  POR ... W:42 L:40 PTS:115.5 +/-:+0.0
  PHX ... W:45 L:37 PTS:112.6 +/-:+0.0

Computing H2H stats ...

Team   Seed Conf    W   L    PTS    FG%    3P%   REB   AST   STL   BLK   TOV    +/-
--------------------------------------------------------------------------------
DET       1 East   60  22  117.8  0.487  0.356  45.6  27.8  10.4   6.4  14.2   +0.0
BOS       2 East   56  26  114.9  0.468  0.367  46.4  24.6   7.1   

In [19]:
import pickle, random
import numpy as np

with open(MODEL_FILE, "rb") as f:
    saved     = pickle.load(f)
ml_model  = saved["model"]
ml_scaler = saved["scaler"]

# ── Win probability ───────────────────────────────────────────────────────────
def win_probability(abbrev_a, abbrev_b):
    """
    P(team A wins a single game vs team B).

    Two inputs to the model:
      1. Season stat differential (A avg - B avg) for all FEAT_KEYS
      2. H2H stat differential for games they actually played each other

    These are averaged so both season-wide dominance AND head-to-head
    performance influence the prediction — just like your disease model
    using multiple patient features together.

    Seed differential is a small additive adjustment on top (~1% per seed gap).
    NO hard clamp to 0.70 — natural model output stays in realistic range.
    """
    avg_a = team_data[abbrev_a]["avg"]
    avg_b = team_data[abbrev_b]["avg"]
    h2h_a = team_data[abbrev_a]["h2h"].get(abbrev_b, {}) or avg_a
    h2h_b = team_data[abbrev_b]["h2h"].get(abbrev_a, {}) or avg_b

    # Season-wide feature
    season_feat = build_feature(avg_a, avg_b)

    # H2H feature (how they did specifically against each other)
    h2h_feat = build_feature(h2h_a, h2h_b)

    # Blend: 70% season, 30% H2H
    blended = 0.70 * season_feat + 0.30 * h2h_feat
    blended = np.nan_to_num(blended).reshape(1, -1)

    feat_scaled = ml_scaler.transform(blended)
    prob = ml_model.predict_proba(feat_scaled)[0][1]

    # Small seed adjustment: each seed gap = ~1% per game
    # (e.g. 1-seed vs 8-seed = +7% boost for the 1-seed)
    seed_a    = seed_map.get(abbrev_a, 4)
    seed_b    = seed_map.get(abbrev_b, 5)
    seed_adj  = (seed_b - seed_a) * 0.01
    prob      = float(np.clip(prob + seed_adj, 0.30, 0.72))
    return prob


# ── Series simulator ──────────────────────────────────────────────────────────
def simulate_series(abbrev_a, abbrev_b):
    p_a      = win_probability(abbrev_a, abbrev_b)
    wins_needed = 4
    wa = wb  = 0
    game_log = []

    while wa < wins_needed and wb < wins_needed:
        noise   = random.gauss(0, 0.035)
        final_p = float(np.clip(p_a + noise, 0.10, 0.90))
        a_wins  = random.random() < final_p
        if a_wins: wa += 1
        else:      wb += 1
        game_log.append({
            "game":   len(game_log) + 1,
            "winner": abbrev_a if a_wins else abbrev_b,
            f"{abbrev_a}_W": wa,
            f"{abbrev_b}_W": wb,
        })

    winner = abbrev_a if wa > wb else abbrev_b
    loser  = abbrev_b if wa > wb else abbrev_a
    ww, lw = (wa, wb) if wa > wb else (wb, wa)
    p_w    = p_a if winner == abbrev_a else 1 - p_a
    return {"winner": winner, "loser": loser, "ww": ww, "lw": lw,
            "games": len(game_log), "p_w": p_w, "log": game_log}


def print_series(a, b, r):
    print(f"\n  🏀 {a} vs {b}")
    print(f"     → {r['winner']} wins {r['ww']}-{r['lw']}  (win prob: {r['p_w']*100:.0f}%)")
    print(f"     {'Game':<8} {'Winner':<8} Series")
    for g in r["log"]:
        print(f"     Game {g['game']:<4} {g['winner']:<8} {g[f'{a}_W']}-{g[f'{b}_W']}")


def simulate_conference(conf_name, seeds):
    print(f"\n{'='*55}\n  {conf_name}\n{'='*55}")
    current     = seeds[:]
    round_names = ["FIRST ROUND", "SEMIFINALS", "CONFERENCE FINALS"]
    for rnd in round_names:
        print(f"\n  ── {rnd} ──")
        n        = len(current)
        matchups = [(current[i], current[n-1-i]) for i in range(n // 2)]
        winners  = []
        for a, b in matchups:
            r = simulate_series(a, b)
            print_series(a, b, r)
            winners.append(r["winner"])
        current = winners
    return current[0]


# ── Run ───────────────────────────────────────────────────────────────────────
print("\n" + "🏀"*20)
print("       NBA PLAYOFF SIMULATION")
print("🏀"*20)

east_champ = simulate_conference("EASTERN CONFERENCE", EAST_SEEDS)
west_champ = simulate_conference("WESTERN CONFERENCE", WEST_SEEDS)

print(f"\n{'='*55}\n  NBA FINALS\n{'='*55}")
r = simulate_series(east_champ, west_champ)
print_series(east_champ, west_champ, r)
champion = r["winner"]

print(f"\n{'🏆'*20}")
print(f"  NBA CHAMPION: {champion}")
print(f"{'🏆'*20}")


🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀
       NBA PLAYOFF SIMULATION
🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀🏀

  EASTERN CONFERENCE

  ── FIRST ROUND ──

  🏀 DET vs ORL
     → DET wins 4-2  (win prob: 72%)
     Game     Winner   Series
     Game 1    DET      1-0
     Game 2    DET      2-0
     Game 3    DET      3-0
     Game 4    ORL      3-1
     Game 5    ORL      3-2
     Game 6    DET      4-2

  🏀 BOS vs PHI
     → BOS wins 4-1  (win prob: 72%)
     Game     Winner   Series
     Game 1    BOS      1-0
     Game 2    BOS      2-0
     Game 3    PHI      2-1
     Game 4    BOS      3-1
     Game 5    BOS      4-1

  🏀 NYK vs ATL
     → NYK wins 4-0  (win prob: 72%)
     Game     Winner   Series
     Game 1    NYK      1-0
     Game 2    NYK      2-0
     Game 3    NYK      3-0
     Game 4    NYK      4-0

  🏀 CLE vs TOR
     → TOR wins 4-1  (win prob: 70%)
     Game     Winner   Series
     Game 1    TOR      0-1
     Game 2    TOR      0-2
     Game 3    TOR      0-3
     Game 4    CLE      1-3
     Game 5    TOR  